In [7]:
import numpy as np
import xarray as xr
import owncloud
from pathlib import Path
import pandas as pd

import torch
import torch.nn as nn

In [2]:
Path('data').mkdir(exist_ok=True, parents=True)

owncloud.Client.from_public_link('https://uni-bonn.sciebo.de/s/3Uf2gScrvuTPQhB').get_file('/', f'data/steinmetz_2017-01-08_Muller.nc')


True

In [4]:
dset = xr.load_dataset('data/steinmetz_2017-01-08_Muller.nc')
dset

<xarray.Dataset> Size: 124MB
Dimensions:             (trial: 261, time: 250, cell: 1268, sample: 82,
                         waveform_component: 3, probe: 384, brain_area_lfp: 5,
                         spike_id: 1836009)
Coordinates:
  * trial               (trial) int32 1kB 1 2 3 4 5 6 ... 257 258 259 260 261
  * time                (time) float64 2kB 0.01 0.02 0.03 0.04 ... 2.48 2.49 2.5
  * cell                (cell) int32 5kB 1 2 3 4 5 ... 1264 1265 1266 1267 1268
  * waveform_component  (waveform_component) int32 12B 1 2 3
  * probe               (probe) int32 2kB 1 2 3 4 5 6 ... 380 381 382 383 384
  * brain_area_lfp      (brain_area_lfp) <U5 100B 'CA1' 'DG' 'LP' 'PO' 'VISam'
  * spike_id            (spike_id) int32 7MB 1 2 3 4 ... 1836007 1836008 1836009
Dimensions without coordinates: sample
Data variables: (12/31)
    contrast_left       (trial) int8 261B 50 0 100 0 50 0 ... 100 0 100 0 100 0
    contrast_right      (trial) int8 261B 0 50 25 100 50 50 ... 100 50 100 25 25
    gocue               (trial) float64 2kB 0.9828 0.902 1.114 ... nan nan nan
    stim_onset          (trial) float64 2kB 0.5 0.5 0.5 0.5 ... 0.5 0.5 0.5 0.5
    feedback_type       (trial) float64 2kB 1.0 1.0 1.0 1.0 ... nan nan nan nan
    feedback_time       (trial) float64 2kB 1.272 1.104 1.402 ... nan nan nan
    ...                  ...
    waveform_w          (cell, sample, waveform_component) float32 1MB 0.0 .....
    waveform_u          (cell, waveform_component, probe) float32 6MB 0.0 ......
    lfp                 (brain_area_lfp, trial, time) float64 3MB -27.6 ... 0...
    spike_time          (spike_id) float32 7MB 2.363 2.385 ... 1.651 0.5142
    spike_cell          (spike_id) uint32 7MB 1 1 1 1 1 ... 1268 1268 1268 1268
    spike_trial         (spike_id) uint32 7MB 1 1 2 2 2 ... 205 205 205 213 252
Attributes:
    session_date:  2017-01-08
    mouse:         Muller
    stim_onset:    0.5
    bin_size:      0.01

Classification

In [ ]:
spike_cols = ['spike_time', 'spike_cell', 'spike_trial']
contrast_cols = ['contrast_left', 'contrast_right']
df_spikes = dset[spike_cols].to_dataframe().reset_index()
df_contrast = dset[contrast_cols].to_dataframe().reset_index()

trial_ids = np.sort(df_spikes['spike_trial'].unique())


<xarray.Dataset> Size: 2kB
Dimensions:         (trial: 261)
Coordinates:
  * trial           (trial) int32 1kB 1 2 3 4 5 6 7 ... 256 257 258 259 260 261
Data variables:
    contrast_left   (trial) int8 261B 50 0 100 0 50 0 0 ... 0 100 0 100 0 100 0
    contrast_right  (trial) int8 261B 0 50 25 100 50 50 ... 50 100 50 100 25 25
Attributes:
    session_date:  2017-01-08
    mouse:         Muller
    stim_onset:    0.5
    bin_size:      0.01

### Train a Classifier to Decode Stimulus Information from Spike Data

Make features out of spike data and labels for trials out of stimulus contrast

In [ ]:

# classify trials based on which side had higher contrast
trial_features = []
trial_labels = []
for trial_id in trial_ids:
    trial_spikes = df_spikes[df_spikes['spike_trial'] == trial_id]
    
    # Count spikes per cell, convert to firing rate
    spikes_per_cell = trial_spikes.groupby('spike_cell').size()
    feature_vector = np.zeros(1268)
    for cell_id, count in spikes_per_cell.items():
        feature_vector[cell_id - 1] = count  # cell_id starts at 1
    
    trial_features.append(feature_vector)
    
    # Label: which side had higher contrast
    left = dset['contrast_left'].values[trial_id - 1]
    right = dset['contrast_right'].values[trial_id - 1]

    if left > right:
        trial_labels.append(0)
    elif right > left:
        trial_labels.append(1)
    else:
        trial_labels.append(2)

Make features and labels tensors

In [152]:
features = torch.tensor(np.array(trial_features), dtype=torch.float32)
labels = torch.tensor(np.array(trial_labels), dtype=torch.long)

Create model

In [151]:
torch.manual_seed(2025)
model = nn.Sequential(
    nn.Linear(1268, 64),
    nn.ReLU(),
    nn.Linear(64, 3)
)

Train

In [154]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(100):
    optimizer.zero_grad()
    labels_pred = model(features)
    loss = loss_fn(labels_pred, labels)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        acc = (torch.argmax(labels_pred, dim=1) == labels).float().mean()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")

Epoch 0, Loss: 1.8470, Accuracy: 0.3908
Epoch 20, Loss: 0.8655, Accuracy: 0.6169
Epoch 40, Loss: 0.5974, Accuracy: 0.8008
Epoch 60, Loss: 0.4064, Accuracy: 0.8774
Epoch 80, Loss: 0.2685, Accuracy: 0.9349


Split into train and test data

In [159]:
from sklearn.model_selection import train_test_split

features_train, features_test, labels_train, labels_test = train_test_split(
    features, labels, train_size=0.8, test_size=0.2, random_state=2025
)

Create model

In [131]:
torch.manual_seed(2025)
model = nn.Sequential(
    nn.Linear(1268, 64),
    nn.ReLU(),
    nn.Linear(64, 3)
)

Train

In [132]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(400):
    optimizer.zero_grad()
    labels_pred = model(features_train)
    loss = loss_fn(labels_pred, labels_train)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        acc = (torch.argmax(labels_pred, dim=1) == labels_train).float().mean()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")

Epoch 0, Loss: 2.2246, Accuracy: 0.3446
Epoch 20, Loss: 0.8574, Accuracy: 0.6231
Epoch 40, Loss: 0.5984, Accuracy: 0.7759
Epoch 60, Loss: 0.4423, Accuracy: 0.8575
Epoch 80, Loss: 0.3252, Accuracy: 0.9106
Epoch 100, Loss: 0.2363, Accuracy: 0.9521
Epoch 120, Loss: 0.1708, Accuracy: 0.9754
Epoch 140, Loss: 0.1235, Accuracy: 0.9845
Epoch 160, Loss: 0.0898, Accuracy: 0.9922
Epoch 180, Loss: 0.0657, Accuracy: 0.9961
Epoch 200, Loss: 0.0490, Accuracy: 1.0000
Epoch 220, Loss: 0.0375, Accuracy: 1.0000
Epoch 240, Loss: 0.0296, Accuracy: 1.0000
Epoch 260, Loss: 0.0240, Accuracy: 1.0000
Epoch 280, Loss: 0.0198, Accuracy: 1.0000
Epoch 300, Loss: 0.0166, Accuracy: 1.0000
Epoch 320, Loss: 0.0141, Accuracy: 1.0000
Epoch 340, Loss: 0.0122, Accuracy: 1.0000
Epoch 360, Loss: 0.0106, Accuracy: 1.0000
Epoch 380, Loss: 0.0093, Accuracy: 1.0000


Test

In [133]:
with torch.no_grad():
    labels_pred = model(features_test)
    acc = (torch.argmax(labels_pred, dim=1) == labels_test).float().mean()
    print(f"Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")

Loss: 0.0083, Accuracy: 0.6477


Create more advanced model

In [150]:
torch.manual_seed(2025)
model = nn.Sequential(
    nn.Linear(1268, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 32),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(32, 3)
)

Train

In [151]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
    optimizer.zero_grad()
    labels_pred = model(features_train)
    loss = loss_fn(labels_pred, labels_train)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        accuracy = (torch.argmax(labels_pred, dim=1) == labels_train).float().mean()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {accuracy.item():.4f}")

Epoch 0, Loss: 1.6053, Accuracy: 0.3510
Epoch 20, Loss: 1.0633, Accuracy: 0.4197
Epoch 40, Loss: 1.0443, Accuracy: 0.4689
Epoch 60, Loss: 0.9441, Accuracy: 0.4935
Epoch 80, Loss: 0.8770, Accuracy: 0.5259
Epoch 100, Loss: 0.7076, Accuracy: 0.6503
Epoch 120, Loss: 0.4224, Accuracy: 0.8355
Epoch 140, Loss: 0.2420, Accuracy: 0.9132
Epoch 160, Loss: 0.1681, Accuracy: 0.9430
Epoch 180, Loss: 0.1019, Accuracy: 0.9611


Test

In [152]:
with torch.no_grad():
    labels_pred = model(features_test)
    acc = (torch.argmax(labels_pred, dim=1) == labels_test).float().mean()
    print(f"Test Accuracy: {acc.item():.4f}")

Test Accuracy: 0.6995


#### Notes:

(Different variants of) more advanced model didn't help with the test accuracy. The dataset is probably too small and the model just memorizes the training data.